In [1]:
import os
import torch
import numpy as np
import pandas as pd
import random
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)


C:\Users\Sidqi\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# 0. SETUP & ULTIMATE SEED
def set_seed_ultimate(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed_ultimate(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Pake Device:", device)

MODEL_NAME = "indobenchmark/indobert-large-p1"
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5
NUM_LABELS = 3
CONFIDENCE_THRESHOLD = 0.95

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


Pake Device: cuda


In [3]:
# ==========================================
# 1. FASE 1: GENERATE LABEL BUAT 400 DATA POOL
# ==========================================
print("\n=== FASE 1: FILTERING DATA POOL DENGAN MODEL 1 ===")

train_manual_df = pd.read_csv("../data_labelling/train_labeled.csv")
pool_df = pd.read_csv("../splitting/pseudo_pool.csv") 

# Load Teacher Model (Model 1)
teacher_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
teacher_model.load_state_dict(torch.load('model1_final.pt'))
teacher_model.to(device)
teacher_model.eval()

confident_texts = []
confident_labels = []

print(f"Total data pool mentah: {len(pool_df)}")
print(f"Menyaring dengan threshold >= {CONFIDENCE_THRESHOLD}...")



=== FASE 1: FILTERING DATA POOL DENGAN MODEL 1 ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-large-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total data pool mentah: 400
Menyaring dengan threshold >= 0.95...


In [4]:
# Looping nebak data pool
with torch.no_grad():
    for text in tqdm(pool_df["cleaned_text"].astype(str)):
        inputs = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt").to(device)
        outputs = teacher_model(**inputs)
        
        # Hitung probabilitas (softmax)
        probs = F.softmax(outputs.logits, dim=1)
        max_prob, pred_label = torch.max(probs, dim=1)
        
        # Kalau model yakin banget (probabilitas >= 0.90), kita ambil!
        if max_prob.item() >= CONFIDENCE_THRESHOLD:
            confident_texts.append(text)
            confident_labels.append(pred_label.item())

# Bikin dataframe buat pseudo yang lolos
pseudo_passed_df = pd.DataFrame({
    "cleaned_text": confident_texts,
    "label": confident_labels
})

print(f"✅ Data pool yang lolos seleksi: {len(pseudo_passed_df)} dari {len(pool_df)} data.")

100%|██████████| 400/400 [01:58<00:00,  3.37it/s]

✅ Data pool yang lolos seleksi: 106 dari 400 data.


In [5]:
# 1100 Manual + X Pseudo yang lolos
hybrid_train_df = pd.concat([train_manual_df, pseudo_passed_df], ignore_index=True)
hybrid_train_df["label"] = hybrid_train_df["label"].astype(int)

print("\nDistribusi Data Hybrid (Model 3):")
print(hybrid_train_df["label"].value_counts())


Distribusi Data Hybrid (Model 3):
label
0    659
2    333
1    214
Name: count, dtype: int64


In [6]:
# ==========================================
# 2. FASE 2: TRAINING MODEL 3 (HYBRID) DARI NOL
# ==========================================
print("\n=== FASE 2: TRAINING MODEL HYBRID ===")

val_df  = pd.read_csv("../data_labelling/val_labeled.csv")
test_df = pd.read_csv("../data_labelling/test_labeled.csv")
val_df["label"]  = val_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

class SentimentDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["cleaned_text"].values
        self.labels = dataframe["label"].values
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = SentimentDataset(hybrid_train_df) # Pake data gabungan!
val_dataset   = SentimentDataset(val_df)
test_dataset  = SentimentDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)


=== FASE 2: TRAINING MODEL HYBRID ===


In [7]:
# Panggil Model Baru (Bener-bener fresh/nol)
student_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
student_model.to(device)

class_weights = compute_class_weight(class_weight="balanced", classes=hybrid_train_df["label"].unique(), y=hybrid_train_df["label"])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW(student_model.parameters(), lr=LR)
ACCUMULATION_STEPS = 4
total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

best_val_loss = float('inf') 
MODEL3_SAVE_PATH = 'model3_uc2.pt' # Pake nama baru biar Model 1 gak ketimpa!

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-large-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
for epoch in range(EPOCHS):
    student_model.train()
    total_train_loss = 0
    loop = tqdm(train_loader, leave=True)
    optimizer.zero_grad()
    
    for step, batch in enumerate(loop):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = student_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels) / ACCUMULATION_STEPS
        loss.backward()
        total_train_loss += loss.item() * ACCUMULATION_STEPS
        
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item() * ACCUMULATION_STEPS)
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    student_model.eval()
    total_val_loss = 0
    val_preds, val_labels = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = student_model(input_ids=input_ids, attention_mask=attention_mask)
            
            loss = criterion(outputs.logits, labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(outputs.logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"\nEpoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(student_model.state_dict(), MODEL3_SAVE_PATH)
        print(f"🔥 Model 3 (Hybrid) membaik! Disimpan ke {MODEL3_SAVE_PATH}")



Epoch 1: 100%|██████████| 151/151 [21:50<00:00,  8.68s/it, loss=1.13] 



Epoch 1 | Train Loss: 1.0246 | Val Loss: 0.9319
🔥 Model 3 (Hybrid) membaik! Disimpan ke model3_uc2.pt


Epoch 2: 100%|██████████| 151/151 [18:45<00:00,  7.46s/it, loss=0.473]



Epoch 2 | Train Loss: 0.6689 | Val Loss: 0.7567
🔥 Model 3 (Hybrid) membaik! Disimpan ke model3_uc2.pt


Epoch 3: 100%|██████████| 151/151 [18:44<00:00,  7.45s/it, loss=0.245] 



Epoch 3 | Train Loss: 0.3713 | Val Loss: 0.7335
🔥 Model 3 (Hybrid) membaik! Disimpan ke model3_uc2.pt


Epoch 4: 100%|██████████| 151/151 [18:44<00:00,  7.45s/it, loss=0.198] 



Epoch 4 | Train Loss: 0.2153 | Val Loss: 0.7977


Epoch 5: 100%|██████████| 151/151 [24:02<00:00,  9.56s/it, loss=0.0715] 



Epoch 5 | Train Loss: 0.1320 | Val Loss: 0.8083


In [ ]:
# ==========================================
# 3. TESTING MODEL 3 (THE MOMENT OF TRUTH)
# ==========================================
print("\n=== HASIL AKHIR MODEL 3 (HYBRID) ===")
student_model.load_state_dict(torch.load(MODEL3_SAVE_PATH))
student_model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = student_model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        preds = torch.argmax(outputs.logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(batch["labels"].cpu().numpy())

print(classification_report(test_labels, test_preds))
print("Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

# Jangan lupa save model finalnya kalau lu butuh deploy
# student_model.save_pretrained("./model3_hybrid_final")
# tokenizer.save_pretrained("./model3_hybrid_final")


=== HASIL AKHIR MODEL 3 (HYBRID) ===
              precision    recall  f1-score   support

           0       0.85      0.75      0.80       130
           1       0.69      0.60      0.64        60
           2       0.59      0.82      0.69        60

    accuracy                           0.73       250
   macro avg       0.71      0.72      0.71       250
weighted avg       0.75      0.73      0.73       250

Confusion Matrix:
[[98 10 22]
 [12 36 12]
 [ 5  6 49]]


: 